# SkySat complaint box: LoRA training on Colab

Trains a LoRA adapter for **Qwen2.5-1.5B-Instruct** that reads a real airline review and answers with a score out of 10, recommended yes/no and the main complaint. It gates the adapter, logs it to the team's MLflow (DagsHub) and registers it as `skysat-reviews`. The server downloads the new `champion` adapter by itself.

1. **Runtime > Change runtime type > T4 GPU.**
2. Add these in **Secrets** (key icon, left bar) and switch on notebook access for each: `KAGGLE_USERNAME`, `KAGGLE_KEY` (kaggle.com > Settings > API), `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (the same values as in the repo's `.env`).
3. **Runtime > Run all.** About 15-20 minutes on a T4.

The dataset (`joelljungstrom/128k-airline-reviews`, licence DbCL-1.0) is downloaded into this Colab machine only. Nothing from it (reviews, prompts, generated text) is saved to Drive, GitHub or MLflow.

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
BRANCH = "main"  # the branch whose code trains
!rm -rf skysat-mlops && git clone -q --depth 1 -b {BRANCH} https://github.com/deensza/skysat-mlops.git
%cd skysat-mlops
!git log -1 --oneline
!pip install -q -r requirements-llm.txt kaggle

## 2. Secrets and data

In [ ]:
import os
from google.colab import userdata
for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    os.environ[key] = userdata.get(key)
!kaggle datasets download joelljungstrom/128k-airline-reviews -p data/reviews --unzip

## 3. Before retraining: how does the current champion do on a later year?

`SPLIT = "live-2020"` scores the champion on the reviews flown in 2020 (the model was trained on reviews flown up to 2018). Skip this cell if there is no champion yet.

In [ ]:
SPLIT = "live-2020"   # use "train" for the first model
!python -m llm.train_lora --eval-only --split {SPLIT}

## 4. Train, gate, log and register

`SPLIT = "train"` trains on reviews flown up to 2018 and gates on the validation reviews; `live-2020` retrains on 2020 and gates on 10% of that year. Registered only if the gate passes (limits in `llm/gate.py`, PROVISIONAL); becomes `champion` only if its `combined` score beats the current champion on the same reviews.

In [ ]:
!python -m llm.train_lora --split {SPLIT}